# Petals to the Metal — Flower Classification

Kaggle competition: [tpu-getting-started](https://www.kaggle.com/c/tpu-getting-started)

## Pre-processing

The dataset ships pre-resized at 192, 224, 331, and 512 px. We use the **224×224** TFRecords (the input size ResNet50 / MobileNet expect), so resizing is done by choosing that folder. Each image is decoded from JPEG and its pixel values are standardized to the range [0, 1].

In [1]:
import re
from pathlib import Path

import tensorflow as tf

IMAGE_SIZE = [224, 224]
BATCH_SIZE = 32
AUTO = tf.data.AUTOTUNE

DATA_DIR = Path("data/tpu-getting-started/tfrecords-jpeg-224x224")
TRAIN_FILENAMES = sorted(str(p) for p in (DATA_DIR / "train").glob("*.tfrec"))
VALID_FILENAMES = sorted(str(p) for p in (DATA_DIR / "val").glob("*.tfrec"))
TEST_FILENAMES = sorted(str(p) for p in (DATA_DIR / "test").glob("*.tfrec"))

print("TensorFlow", tf.__version__)
print(len(TRAIN_FILENAMES), "train files,", len(VALID_FILENAMES), "val files,", len(TEST_FILENAMES), "test files")

TensorFlow 2.21.0
16 train files, 16 val files, 16 test files


In [2]:
CLASSES = [
    'pink primrose', 'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea', 'wild geranium',
    'tiger lily', 'moon orchid', 'bird of paradise', 'monkshood', 'globe thistle',                # 00 - 09
    'snapdragon', "colt's foot", 'king protea', 'spear thistle', 'yellow iris',
    'globe-flower', 'purple coneflower', 'peruvian lily', 'balloon flower', 'giant white arum lily', # 10 - 19
    'fire lily', 'pincushion flower', 'fritillary', 'red ginger', 'grape hyacinth',
    'corn poppy', 'prince of wales feathers', 'stemless gentian', 'artichoke', 'sweet william',     # 20 - 29
    'carnation', 'garden phlox', 'love in the mist', 'cosmos', 'alpine sea holly',
    'ruby-lipped cattleya', 'cape flower', 'great masterwort', 'siam tulip', 'lenten rose',         # 30 - 39
    'barberton daisy', 'daffodil', 'sword lily', 'poinsettia', 'bolero deep blue',
    'wallflower', 'marigold', 'buttercup', 'daisy', 'common dandelion',                            # 40 - 49
    'petunia', 'wild pansy', 'primula', 'sunflower', 'lilac hibiscus',
    'bishop of llandaff', 'gaura', 'geranium', 'orange dahlia', 'pink-yellow dahlia',              # 50 - 59
    'cautleya spicata', 'japanese anemone', 'black-eyed susan', 'silverbush', 'californian poppy',
    'osteospermum', 'spring crocus', 'iris', 'windflower', 'tree poppy',                           # 60 - 69
    'gazania', 'azalea', 'water lily', 'rose', 'thorn apple',
    'morning glory', 'passion flower', 'lotus', 'toad lily', 'anthurium',                          # 70 - 79
    'frangipani', 'clematis', 'hibiscus', 'columbine', 'desert-rose',
    'tree mallow', 'magnolia', 'cyclamen ', 'watercress', 'canna lily',                            # 80 - 89
    'hippeastrum ', 'bee balm', 'pink quill', 'foxglove', 'bougainvillea',
    'camellia', 'mallow', 'mexican petunia', 'bromelia', 'blanket flower',                         # 90 - 99
    'trumpet creeper', 'blackberry lily', 'common tulip', 'wild rose',                             # 100 - 103
]
len(CLASSES)

104

In [3]:
def decode_image(image_data):
    image = tf.image.decode_jpeg(image_data, channels=3)
    image = tf.cast(image, tf.float32) / 255.0
    return tf.reshape(image, [*IMAGE_SIZE, 3])


def read_labeled_tfrecord(example):
    features = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "class": tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example, features)
    return decode_image(example["image"]), tf.cast(example["class"], tf.int32)


def read_unlabeled_tfrecord(example):
    features = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "id": tf.io.FixedLenFeature([], tf.string),
    }
    example = tf.io.parse_single_example(example, features)
    return decode_image(example["image"]), example["id"]


def load_dataset(filenames, labeled=True, ordered=False):
    options = tf.data.Options()
    options.deterministic = ordered
    dataset = tf.data.TFRecordDataset(filenames, num_parallel_reads=AUTO).with_options(options)
    parser = read_labeled_tfrecord if labeled else read_unlabeled_tfrecord
    return dataset.map(parser, num_parallel_calls=AUTO)


def count_images(filenames):
    # Filenames end in the number of images in the file, e.g. 00-224x224-798.tfrec
    return sum(int(re.search(r"-(\d+)\.tfrec$", f).group(1)) for f in filenames)

In [4]:
train_ds = load_dataset(TRAIN_FILENAMES).batch(BATCH_SIZE).prefetch(AUTO)
valid_ds = load_dataset(VALID_FILENAMES, ordered=True).batch(BATCH_SIZE).prefetch(AUTO)
test_ds = load_dataset(TEST_FILENAMES, labeled=False, ordered=True).batch(BATCH_SIZE).prefetch(AUTO)

print("Train images:", count_images(TRAIN_FILENAMES))
print("Val images:  ", count_images(VALID_FILENAMES))
print("Test images: ", count_images(TEST_FILENAMES))

Train images: 12753
Val images:   3712
Test images:  7382


In [5]:
images, labels = next(iter(train_ds))
print("Batch shape:", images.shape, images.dtype)
print(f"Pixel range: {float(tf.reduce_min(images)):.3f} to {float(tf.reduce_max(images)):.3f}")
print("First labels:", [CLASSES[i] for i in labels[:5].numpy()])

Batch shape: (32, 224, 224, 3) <dtype: 'float32'>
Pixel range: 0.000 to 1.000
First labels: ['geranium', 'primula', 'wild pansy', 'snapdragon', 'fire lily']


I0000 00:00:1791316882.467971 6006895 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144
